In [2]:
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime

# Используем точный путь к файлу
file_path = r"C:\Users\Мартинсон Диана\cars\data\raw\avito_cars.csv"

# Проверяем существование файла
if os.path.exists(file_path):
    print(f"✅ Файл найден: {file_path}")
    
    # Загружаем данные
    df = pd.read_csv(file_path)
    print(f"📖 Загружено записей: {len(df)}")
    print("\nСырые данные:")
    print(df.info())
    print("\nПервые 10 строк:")
    print(df.head(10))
    
    # Покажем названия столбцов
    print("\n📋 Столбцы в данных:")
    for i, col in enumerate(df.columns, 1):
        print(f"{i}. {col}")
        
else:
    print(f"❌ Файл не найден: {file_path}")
    print("Проверьте путь и наличие файла")

✅ Файл найден: C:\Users\Мартинсон Диана\cars\data\raw\avito_cars.csv
📖 Загружено записей: 50

Сырые данные:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   location      50 non-null     object
 1   price         50 non-null     object
 2   price_number  50 non-null     object
 3   params        50 non-null     object
 4   id            50 non-null     int64 
 5   description   50 non-null     object
 6   title         50 non-null     object
 7   url           50 non-null     object
 8   date          19 non-null     object
 9   engine        50 non-null     object
 10  mileage       50 non-null     object
dtypes: int64(1), object(10)
memory usage: 4.4+ KB
None

Первые 10 строк:
                                            location       price price_number  \
0  Краснодарский край, Курганинский р-н, Курганин...    599 000₽      599 000   
1      

In [3]:
print("\nНАЧИНАЕМ ОЧИСТКУ...")

df_clean = df.copy()

initial_count = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Удалено дубликатов: {initial_count - len(df_clean)}")

prices = []
for price_str in df_clean['price_number']:
    try:
        clean_price = re.sub(r'[^\d]', '', str(price_str).replace('\xa0', ''))
        if clean_price:
            prices.append(float(clean_price))
        else:
            prices.append(np.nan)
    except:
        prices.append(np.nan)

df_clean['price'] = prices

df_clean = df_clean.dropna(subset=['price'])
print(f"Записей с ценой: {len(df_clean)}")

df_clean = df_clean[(df_clean['price'] >= 50000) & (df_clean['price'] <= 50000000)]
print(f"Записей после фильтрации по цене: {len(df_clean)}")

if len(df_clean) > 0:
    print("\nСТАТИСТИКА:")
    print(f"Минимальная цена: {df_clean['price'].min():.0f} руб.")
    print(f"Максимальная цена: {df_clean['price'].max():.0f} руб.")
    print(f"Средняя цена: {df_clean['price'].mean():.0f} руб.")
    print(f"Медианная цена: {df_clean['price'].median():.0f} руб.")
else:
    print("Нет данных после очистки")

print(f"\nСтолбцы в данных: {list(df_clean.columns)}")


НАЧИНАЕМ ОЧИСТКУ...
Удалено дубликатов: 0
Записей с ценой: 50
Записей после фильтрации по цене: 50

СТАТИСТИКА:
Минимальная цена: 180000 руб.
Максимальная цена: 2650000 руб.
Средняя цена: 852520 руб.
Медианная цена: 644500 руб.

Столбцы в данных: ['location', 'price', 'price_number', 'params', 'id', 'description', 'title', 'url', 'date', 'engine', 'mileage']


In [4]:
print("ИЗВЛЕКАЕМ ПРИЗНАКИ...")

def extract_brand(title):
    if pd.isna(title):
        return "Unknown"
    title_str = str(title).lower()
    
    brands = ['bmw', 'mercedes', 'audi', 'volkswagen', 'toyota', 'honda', 
              'nissan', 'hyundai', 'kia', 'ford', 'chevrolet', 'renault',
              'lada', 'skoda', 'mazda', 'mitsubishi', 'lexus', 'subaru',
              'volvo', 'opel', 'peugeot', 'citroen']
    
    for brand in brands:
        if brand in title_str:
            return brand.title()
    
    words = str(title).split()
    if words:
        return words[0]
    return "Unknown"

df_clean['brand'] = df_clean['title'].apply(extract_brand)
print(f"Уникальных марок: {df_clean['brand'].nunique()}")

def extract_year(row):
    if 'params' in row and pd.notna(row['params']):
        params_str = str(row['params'])
        year_match = re.search(r'\b(19|20)\d{2}\b', params_str)
        if year_match:
            return int(year_match.group())
    
    if 'title' in row and pd.notna(row['title']):
        title_str = str(row['title'])
        year_match = re.search(r'\b(19|20)\d{2}\b', title_str)
        if year_match:
            return int(year_match.group())
    
    return np.nan

df_clean['year'] = df_clean.apply(extract_year, axis=1)
print(f"Записей с годом: {df_clean['year'].notna().sum()}")

current_year = datetime.now().year
df_clean['age'] = current_year - df_clean['year']

df_clean = df_clean[(df_clean['year'] >= 1990) & (df_clean['year'] <= current_year)]
print(f"После фильтрации по году: {len(df_clean)}")

ИЗВЛЕКАЕМ ПРИЗНАКИ...
Уникальных марок: 21
Записей с годом: 50
После фильтрации по году: 50


In [5]:
print("СОХРАНЯЕМ РЕЗУЛЬТАТЫ...")

processed_dir = r"C:\Users\Мартинсон Диана\cars\data\processed"
os.makedirs(processed_dir, exist_ok=True)

output_path = os.path.join(processed_dir, "cleaned_cars.csv")
df_clean.to_csv(output_path, index=False)

print("=" * 50)
print("ОЧИСТКА ЗАВЕРШЕНА!")
print(f"Финальное количество записей: {len(df_clean)}")
print(f"Сохранено в: {output_path}")

print(f"\nФИНАЛЬНАЯ СТАТИСТИКА:")
print(f"Средняя цена: {df_clean['price'].mean():.0f} руб.")
print(f"Уникальных марок: {df_clean['brand'].nunique()}")
if 'year' in df_clean.columns:
    valid_years = df_clean['year'].notna().sum()
    if valid_years > 0:
        print(f"Годы: {df_clean['year'].min()} - {df_clean['year'].max()}")

print(f"\nТОП-5 МАРОК:")
brand_counts = df_clean['brand'].value_counts().head(5)
for brand, count in brand_counts.items():
    print(f"  {brand}: {count} авто")

СОХРАНЯЕМ РЕЗУЛЬТАТЫ...
ОЧИСТКА ЗАВЕРШЕНА!
Финальное количество записей: 50
Сохранено в: C:\Users\Мартинсон Диана\cars\data\processed\cleaned_cars.csv

ФИНАЛЬНАЯ СТАТИСТИКА:
Средняя цена: 852520 руб.
Уникальных марок: 21
Годы: 1998 - 2024

ТОП-5 МАРОК:
  Lada: 7 авто
  Hyundai: 7 авто
  Ford: 4 авто
  Kia: 3 авто
  Opel: 3 авто
